# 🥈 Silver Layer Setup — Reference Data & Infrastructure

This notebook sets up the Silver layer infrastructure including:
- Silver schema creation
- Lookup tables (status codes, payment methods, loyalty tiers, weight units)
- Bridge tables (category taxonomy mapping)
- Data quality logging table

---

## Purpose
This notebook is designed to be run **once initially**, then periodically when:
- New status codes need to be added
- Category mappings change
- Business rules are updated

All operations are **idempotent** — safe to re-run without side effects.

---

## Execution Frequency
- **Initial setup**: Run once before first Silver load
- **Maintenance**: Weekly or when reference data changes
- **Not incremental**: Full refresh of lookup tables each run

---

## Design Principles
- **Configuration over code** — all mappings are data-driven
- **Single source of truth** — lookup tables are referenced by all Silver transformations
- **Snowpark-first** — uses DataFrames where possible, SQL for DDL only
- **Idempotent** — CREATE OR REPLACE ensures safe re-runs

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Imports & Session Setup
# ══════════════════════════════════════════════════════════════════════════════

from datetime import datetime
from snowflake.snowpark import Session
from snowflake.snowpark.functions import lit, col
from snowflake.snowpark.types import StructType, StructField, StringType, IntegerType, DecimalType, BooleanType

# Get active Snowflake session
session = get_active_session()

print("=" * 70)
print("SILVER LAYER SETUP — Reference Data & Infrastructure")
print("=" * 70)
print(f"Database  : {session.get_current_database()}")
print(f"Schema    : {session.get_current_schema()}")
print(f"Warehouse : {session.get_current_warehouse()}")
print(f"Timestamp : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

---

## Configuration

Define schema name and table structures.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Configuration
# ══════════════════════════════════════════════════════════════════════════════

SILVER_SCHEMA = "SILVER"

print(f"Target Schema: {SILVER_SCHEMA}")
print(f"Setup will create 8 infrastructure tables:")
print(f"  - 5 Lookup tables (LKP_*)")
print(f"  - 2 Bridge tables (BRIDGE_*)")
print(f"  - 1 Data quality log (DQ_LOG)")

---

## Step 1: Create Silver Schema

Creates the Silver schema if it doesn't already exist.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Create Silver Schema
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 70)
print("Creating Silver Schema...")
print("─" * 70)

session.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}").collect()

print(f"✓ Schema '{SILVER_SCHEMA}' ready")

---

## Step 2: Create & Populate Lookup Tables

### 2.1 LKP_ORDER_STATUS

Maps source-specific status codes to canonical values across 4 order sources.

**Purpose**: Standardize order status across Web, Mobile, Wholesale, and Marketplace systems.

**Source Systems**:
- Web Store: Single-char codes (C, P, S, X, R)
- Mobile App: Verbose strings (COMPLETE, PENDING, DISPATCHED, etc.)
- Wholesale: Prefixed codes (ORD_FULFILLED, ORD_PENDING, etc.)
- Marketplace: Titlecase (Shipped, Pending, Cancelled, etc.)

**Canonical Values**: completed, processing, shipped, cancelled, returned

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LKP_ORDER_STATUS
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 70)
print("Creating LKP_ORDER_STATUS...")
print("─" * 70)

# Drop and recreate for clean state
session.sql(f"""
CREATE OR REPLACE TABLE {SILVER_SCHEMA}.LKP_ORDER_STATUS (
    source_system       VARCHAR(20)  NOT NULL,
    source_status_code  VARCHAR(50)  NOT NULL,
    canonical_status    VARCHAR(20)  NOT NULL,
    description         VARCHAR(200),
    PRIMARY KEY (source_system, source_status_code)
)
""").collect()

# Define all status mappings
status_mappings = [
    # Web Store mappings
    ('web', 'C', 'completed', 'Order completed and delivered'),
    ('web', 'P', 'processing', 'Order is being processed'),
    ('web', 'S', 'shipped', 'Order has been shipped'),
    ('web', 'X', 'cancelled', 'Order was cancelled'),
    ('web', 'R', 'returned', 'Order was returned'),
    
    # Mobile App mappings
    ('mobile', 'COMPLETE', 'completed', 'Order completed and delivered'),
    ('mobile', 'PENDING', 'processing', 'Order is being processed'),
    ('mobile', 'DISPATCHED', 'shipped', 'Order has been dispatched'),
    ('mobile', 'CANCELED', 'cancelled', 'Order was canceled'),
    ('mobile', 'REFUND_REQUESTED', 'returned', 'Refund has been requested'),
    
    # Wholesale mappings
    ('wholesale', 'ORD_FULFILLED', 'completed', 'Order fulfilled'),
    ('wholesale', 'ORD_PENDING', 'processing', 'Order pending fulfillment'),
    ('wholesale', 'ORD_SHIPPED', 'shipped', 'Order shipped to customer'),
    ('wholesale', 'ORD_VOIDED', 'cancelled', 'Order voided'),
    ('wholesale', 'ORD_RETURNED', 'returned', 'Order returned'),
    
    # Marketplace mappings
    ('marketplace', 'Shipped', 'shipped', 'Order shipped via Amazon'),
    ('marketplace', 'Pending', 'processing', 'Order pending shipment'),
    ('marketplace', 'Cancelled', 'cancelled', 'Order cancelled'),
    ('marketplace', 'Returned', 'returned', 'Order returned'),
]

# Create DataFrame and insert
schema = StructType([
    StructField("source_system", StringType()),
    StructField("source_status_code", StringType()),
    StructField("canonical_status", StringType()),
    StructField("description", StringType())
])

status_df = session.create_dataframe(status_mappings, schema=schema)
status_df.write.mode("append").save_as_table(f"{SILVER_SCHEMA}.LKP_ORDER_STATUS")

row_count = session.table(f"{SILVER_SCHEMA}.LKP_ORDER_STATUS").count()
print(f"✓ LKP_ORDER_STATUS created with {row_count} mappings")

# Show sample
print("\nSample mappings:")
session.table(f"{SILVER_SCHEMA}.LKP_ORDER_STATUS").limit(5).show()

---

### 2.2 LKP_PAYMENT_STATUS

Decodes numeric payment status codes from the payment gateway.

**Source**: BRONZE.PAY__PAYMENT_STATUS_LOOKUP (loaded from payment_status_lookup.csv)

**Status Codes**:
- 1 = succeeded
- 2 = pending
- 3 = failed
- 4 = refunded

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LKP_PAYMENT_STATUS
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 70)
print("Creating LKP_PAYMENT_STATUS...")
print("─" * 70)

# Drop and recreate
session.sql(f"""
CREATE OR REPLACE TABLE {SILVER_SCHEMA}.LKP_PAYMENT_STATUS (
    status_code     INTEGER      NOT NULL PRIMARY KEY,
    status_name     VARCHAR(20)  NOT NULL,
    is_successful   BOOLEAN      NOT NULL
)
""").collect()

# Load from Bronze lookup table if it exists
try:
    print("  Attempting to load from BRONZE.PAY__PAYMENT_STATUS_LOOKUP...")
    bronze_lookup = session.table("BRONZE.PAY__PAYMENT_STATUS_LOOKUP")
    
    # Transform and insert - use quoted column names from Bronze
    payment_status_df = bronze_lookup.select(
        col('"stat_cd"').cast(IntegerType()).alias("status_code"),
        col('"stat_desc"').cast(StringType()).alias("status_name"),
        # Mark succeeded (1) and refunded (4) as successful
        (col('"stat_cd"').in_(lit(1), lit(4))).alias("is_successful")
    )
    
    payment_status_df.write.mode("append").save_as_table(f"{SILVER_SCHEMA}.LKP_PAYMENT_STATUS")
    
    row_count = session.table(f"{SILVER_SCHEMA}.LKP_PAYMENT_STATUS").count()
    print(f"✓ LKP_PAYMENT_STATUS created with {row_count} codes from Bronze")
    
except Exception as e:
    # If Bronze table doesn't exist or column names are wrong, insert manually
    print(f"⚠ Could not load from Bronze: {str(e)[:300]}")
    
    # Try to show available columns for debugging
    try:
        bronze_lookup = session.table("BRONZE.PAY__PAYMENT_STATUS_LOOKUP")
        print(f"  Available columns: {[field.name for field in bronze_lookup.schema.fields]}")
    except:
        pass
    
    print(f"  Falling back to manual data insertion...")
    
    payment_mappings = [
        (1, 'succeeded', True),
        (2, 'pending', False),
        (3, 'failed', False),
        (4, 'refunded', True)
    ]
    
    schema = StructType([
        StructField("status_code", IntegerType()),
        StructField("status_name", StringType()),
        StructField("is_successful", BooleanType())
    ])
    
    payment_df = session.create_dataframe(payment_mappings, schema=schema)
    payment_df.write.mode("append").save_as_table(f"{SILVER_SCHEMA}.LKP_PAYMENT_STATUS")
    
    print(f"✓ LKP_PAYMENT_STATUS created with 4 codes (manual)")

# Show all codes
print("\nAll payment status codes:")
session.table(f"{SILVER_SCHEMA}.LKP_PAYMENT_STATUS").show()

---

### 2.3 LKP_PAYMENT_METHOD

Decodes payment method codes from transaction data.

**Source**: Discovered from BRONZE.PAY__TRANSACTIONS.pymt_mthd_cd column

**Payment Method Codes**:
- CC = Credit Card
- DC = Debit Card
- PP = PayPal
- APY = Apple Pay
- GPY = Google Pay
- BT = Bank Transfer
- BNPL = Buy Now Pay Later

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LKP_PAYMENT_METHOD
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 70)
print("Creating LKP_PAYMENT_METHOD...")
print("─" * 70)

session.sql(f"""
CREATE OR REPLACE TABLE {SILVER_SCHEMA}.LKP_PAYMENT_METHOD (
    payment_method_code VARCHAR(10)  NOT NULL PRIMARY KEY,
    payment_method_name VARCHAR(50)  NOT NULL,
    category            VARCHAR(20)  NOT NULL
)
""").collect()

# Define all payment method mappings
payment_methods = [
    ('CC', 'Credit Card', 'card'),
    ('DC', 'Debit Card', 'card'),
    ('PP', 'PayPal', 'wallet'),
    ('APY', 'Apple Pay', 'wallet'),
    ('GPY', 'Google Pay', 'wallet'),
    ('BT', 'Bank Transfer', 'bank_transfer'),
    ('BNPL', 'Buy Now Pay Later', 'bnpl')
]

schema = StructType([
    StructField("payment_method_code", StringType()),
    StructField("payment_method_name", StringType()),
    StructField("category", StringType())
])

payment_method_df = session.create_dataframe(payment_methods, schema=schema)
payment_method_df.write.mode("append").save_as_table(f"{SILVER_SCHEMA}.LKP_PAYMENT_METHOD")

row_count = session.table(f"{SILVER_SCHEMA}.LKP_PAYMENT_METHOD").count()
print(f"✓ LKP_PAYMENT_METHOD created with {row_count} methods")

# Show all methods
print("\nAll payment methods:")
session.table(f"{SILVER_SCHEMA}.LKP_PAYMENT_METHOD").show()

---

### 2.4 LKP_LOYALTY_TIER

Maps loyalty tier codes between Web Store and Mobile App systems.

**Source Systems**:
- Web Store: Abbreviated codes (STD, SLV, GLD, PLT)
- Mobile App: Verbose strings (standard, silver, gold, platinum)

**Canonical Values**: standard, silver, gold, platinum (with ranking 1-4)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LKP_LOYALTY_TIER
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 70)
print("Creating LKP_LOYALTY_TIER...")
print("─" * 70)

session.sql(f"""
CREATE OR REPLACE TABLE {SILVER_SCHEMA}.LKP_LOYALTY_TIER (
    source_system    VARCHAR(20)  NOT NULL,
    source_tier_code VARCHAR(20)  NOT NULL,
    canonical_tier   VARCHAR(20)  NOT NULL,
    tier_rank        INTEGER      NOT NULL,
    PRIMARY KEY (source_system, source_tier_code)
)
""").collect()

# Define all tier mappings
tier_mappings = [
    # Web Store mappings
    ('web', 'STD', 'standard', 1),
    ('web', 'SLV', 'silver', 2),
    ('web', 'GLD', 'gold', 3),
    ('web', 'PLT', 'platinum', 4),
    
    # Mobile App mappings
    ('mobile', 'standard', 'standard', 1),
    ('mobile', 'silver', 'silver', 2),
    ('mobile', 'gold', 'gold', 3),
    ('mobile', 'platinum', 'platinum', 4)
]

schema = StructType([
    StructField("source_system", StringType()),
    StructField("source_tier_code", StringType()),
    StructField("canonical_tier", StringType()),
    StructField("tier_rank", IntegerType())
])

tier_df = session.create_dataframe(tier_mappings, schema=schema)
tier_df.write.mode("append").save_as_table(f"{SILVER_SCHEMA}.LKP_LOYALTY_TIER")

row_count = session.table(f"{SILVER_SCHEMA}.LKP_LOYALTY_TIER").count()
print(f"✓ LKP_LOYALTY_TIER created with {row_count} mappings")

# Show all tiers
print("\nAll loyalty tier mappings:")
session.table(f"{SILVER_SCHEMA}.LKP_LOYALTY_TIER").show()

---

### 2.5 LKP_WEIGHT_UNITS

Weight unit conversion factors for shipment data.

**Source**: BRONZE.SHP__SHIPMENTS uses mixed units (GRM and KGM)

**Conversion**: All weights normalized to kilograms in Silver layer

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LKP_WEIGHT_UNITS
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 70)
print("Creating LKP_WEIGHT_UNITS...")
print("─" * 70)

session.sql(f"""
CREATE OR REPLACE TABLE {SILVER_SCHEMA}.LKP_WEIGHT_UNITS (
    unit_code        VARCHAR(10)     NOT NULL PRIMARY KEY,
    unit_name        VARCHAR(50)     NOT NULL,
    to_kg_multiplier DECIMAL(10, 6)  NOT NULL
)
""").collect()

# Define weight unit conversions
weight_units = [
    ('GRM', 'Grams', 0.001),      # 1 gram = 0.001 kg
    ('KGM', 'Kilograms', 1.0)     # 1 kg = 1.0 kg
]

schema = StructType([
    StructField("unit_code", StringType()),
    StructField("unit_name", StringType()),
    StructField("to_kg_multiplier", DecimalType(10, 6))
])

weight_df = session.create_dataframe(weight_units, schema=schema)
weight_df.write.mode("append").save_as_table(f"{SILVER_SCHEMA}.LKP_WEIGHT_UNITS")

row_count = session.table(f"{SILVER_SCHEMA}.LKP_WEIGHT_UNITS").count()
print(f"✓ LKP_WEIGHT_UNITS created with {row_count} units")

# Show all units
print("\nAll weight units:")
session.table(f"{SILVER_SCHEMA}.LKP_WEIGHT_UNITS").show()

---

## Step 3: Create Bridge Tables

### 3.1 BRIDGE_CATEGORY

Maps category names across all 5 source systems to canonical codes.

**Challenge**: Same product category has 5 different names across sources:
- Electronics: `Electronics` (Web) / `Tech & Gadgets` (Mobile) / `ELECTRONICS` (Wholesale) / `Electronics & Computers` (Marketplace) / `Tech & Gadgets` (Reviews)

**Solution**: Map all variants to canonical code `ELEC` for cross-source analysis

**10 Product Categories**: ELEC, CLTH, HOME, SPRT, BOOK, BEAU, TOYS, FOOD, TOOL, PET

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BRIDGE_CATEGORY
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 70)
print("Creating BRIDGE_CATEGORY...")
print("─" * 70)

session.sql(f"""
CREATE OR REPLACE TABLE {SILVER_SCHEMA}.BRIDGE_CATEGORY (
    category_code   VARCHAR(10)  NOT NULL,
    category_name   VARCHAR(100) NOT NULL,
    source_system   VARCHAR(20)  NOT NULL,
    source_category VARCHAR(100) NOT NULL,
    PRIMARY KEY (category_code, source_system, source_category)
)
""").collect()

# Define all category mappings (10 categories × 5 sources = 50 mappings)
category_mappings = [
    # Electronics (ELEC)
    ('ELEC', 'Electronics', 'web', 'Electronics'),
    ('ELEC', 'Electronics', 'mobile', 'Tech & Gadgets'),
    ('ELEC', 'Electronics', 'wholesale', 'ELECTRONICS'),
    ('ELEC', 'Electronics', 'marketplace', 'Electronics & Computers'),
    ('ELEC', 'Electronics', 'reviews', 'Tech & Gadgets'),
    
    # Clothing & Apparel (CLTH)
    ('CLTH', 'Clothing & Apparel', 'web', 'Clothing & Apparel'),
    ('CLTH', 'Clothing & Apparel', 'mobile', 'Fashion'),
    ('CLTH', 'Clothing & Apparel', 'wholesale', 'APPAREL_CLOTHING'),
    ('CLTH', 'Clothing & Apparel', 'marketplace', 'Clothing, Shoes & Jewelry'),
    ('CLTH', 'Clothing & Apparel', 'reviews', 'Fashion'),
    
    # Home & Kitchen (HOME)
    ('HOME', 'Home & Kitchen', 'web', 'Home & Kitchen'),
    ('HOME', 'Home & Kitchen', 'mobile', 'Home Living'),
    ('HOME', 'Home & Kitchen', 'wholesale', 'HOME_GOODS'),
    ('HOME', 'Home & Kitchen', 'marketplace', 'Home & Kitchen'),
    ('HOME', 'Home & Kitchen', 'reviews', 'Home Living'),
    
    # Sports & Outdoors (SPRT)
    ('SPRT', 'Sports & Outdoors', 'web', 'Sports & Outdoors'),
    ('SPRT', 'Sports & Outdoors', 'mobile', 'Sports & Fitness'),
    ('SPRT', 'Sports & Outdoors', 'wholesale', 'SPORTING_GOODS'),
    ('SPRT', 'Sports & Outdoors', 'marketplace', 'Sports & Outdoors'),
    ('SPRT', 'Sports & Outdoors', 'reviews', 'Sports & Fitness'),
    
    # Books & Media (BOOK)
    ('BOOK', 'Books & Media', 'web', 'Books & Media'),
    ('BOOK', 'Books & Media', 'mobile', 'Books'),
    ('BOOK', 'Books & Media', 'wholesale', 'PUBLICATIONS'),
    ('BOOK', 'Books & Media', 'marketplace', 'Books'),
    ('BOOK', 'Books & Media', 'reviews', 'Books'),
    
    # Beauty & Personal Care (BEAU)
    ('BEAU', 'Beauty & Personal Care', 'web', 'Beauty & Personal Care'),
    ('BEAU', 'Beauty & Personal Care', 'mobile', 'Beauty'),
    ('BEAU', 'Beauty & Personal Care', 'wholesale', 'BEAUTY_PERSONAL'),
    ('BEAU', 'Beauty & Personal Care', 'marketplace', 'Beauty & Personal Care'),
    ('BEAU', 'Beauty & Personal Care', 'reviews', 'Beauty'),
    
    # Toys & Games (TOYS)
    ('TOYS', 'Toys & Games', 'web', 'Toys & Games'),
    ('TOYS', 'Toys & Games', 'mobile', 'Kids & Toys'),
    ('TOYS', 'Toys & Games', 'wholesale', 'TOYS_GAMES'),
    ('TOYS', 'Toys & Games', 'marketplace', 'Toys & Games'),
    ('TOYS', 'Toys & Games', 'reviews', 'Kids & Toys'),
    
    # Food & Grocery (FOOD)
    ('FOOD', 'Food & Grocery', 'web', 'Food & Grocery'),
    ('FOOD', 'Food & Grocery', 'mobile', 'Food & Drink'),
    ('FOOD', 'Food & Grocery', 'wholesale', 'FOOD_BEVERAGE'),
    ('FOOD', 'Food & Grocery', 'marketplace', 'Grocery & Gourmet Food'),
    ('FOOD', 'Food & Grocery', 'reviews', 'Food & Drink'),
    
    # Tools & Hardware (TOOL)
    ('TOOL', 'Tools & Hardware', 'web', 'Tools & Hardware'),
    ('TOOL', 'Tools & Hardware', 'mobile', 'DIY & Tools'),
    ('TOOL', 'Tools & Hardware', 'wholesale', 'TOOLS_HARDWARE'),
    ('TOOL', 'Tools & Hardware', 'marketplace', 'Tools & Home Improvement'),
    ('TOOL', 'Tools & Hardware', 'reviews', 'DIY & Tools'),
    
    # Pet Supplies (PET)
    ('PET', 'Pet Supplies', 'web', 'Pet Supplies'),
    ('PET', 'Pet Supplies', 'mobile', 'Pets'),
    ('PET', 'Pet Supplies', 'wholesale', 'PET_CARE'),
    ('PET', 'Pet Supplies', 'marketplace', 'Pet Supplies'),
    ('PET', 'Pet Supplies', 'reviews', 'Pets')
]

schema = StructType([
    StructField("category_code", StringType()),
    StructField("category_name", StringType()),
    StructField("source_system", StringType()),
    StructField("source_category", StringType())
])

category_df = session.create_dataframe(category_mappings, schema=schema)
category_df.write.mode("append").save_as_table(f"{SILVER_SCHEMA}.BRIDGE_CATEGORY")

row_count = session.table(f"{SILVER_SCHEMA}.BRIDGE_CATEGORY").count()
print(f"✓ BRIDGE_CATEGORY created with {row_count} mappings")

# Show sample by category code
print("\nSample category mappings (ELEC):")
(session.table(f"{SILVER_SCHEMA}.BRIDGE_CATEGORY")
 .filter(col("category_code") == lit("ELEC"))
 .show())

---

### 3.2 BRIDGE_CUSTOMER_IDENTITY

**Note**: This table is populated by the incremental load notebook after customer tables are created.

For now, we just create the structure. It will be rebuilt with each Silver load run to capture new customer identities.

**Purpose**: Map source-specific customer IDs to canonical email addresses for cross-source customer analysis.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BRIDGE_CUSTOMER_IDENTITY (Structure Only)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 70)
print("Creating BRIDGE_CUSTOMER_IDENTITY structure...")
print("─" * 70)

session.sql(f"""
CREATE OR REPLACE TABLE {SILVER_SCHEMA}.BRIDGE_CUSTOMER_IDENTITY (
    canonical_customer_id VARCHAR(200) NOT NULL,
    source_system         VARCHAR(20)  NOT NULL,
    source_customer_id    VARCHAR(100) NOT NULL,
    first_seen_date       DATE,
    last_seen_date        DATE,
    is_active             BOOLEAN      DEFAULT TRUE,
    PRIMARY KEY (canonical_customer_id, source_system, source_customer_id)
)
""").collect()

print("✓ BRIDGE_CUSTOMER_IDENTITY structure created")
print("  (Will be populated by SILVER_INCREMENTAL_LOAD notebook)")

---

## Step 4: Create Data Quality Log

Tracks all data quality issues encountered during Bronze → Silver transformations.

**Issue Types**:
- duplicate
- missing_value
- invalid_format
- orphan_record
- data_type_error
- mapping_failed

**Severity Levels**: info, warning, error

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DQ_LOG (Data Quality Logging)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 70)
print("Creating DQ_LOG...")
print("─" * 70)

session.sql(f"""
CREATE OR REPLACE TABLE {SILVER_SCHEMA}.DQ_LOG (
    log_id              INTEGER AUTOINCREMENT PRIMARY KEY,
    run_timestamp       TIMESTAMP_NTZ  NOT NULL,
    source_system       VARCHAR(20),
    source_table        VARCHAR(100),
    target_table        VARCHAR(100),
    issue_type          VARCHAR(50)    NOT NULL,
    issue_severity      VARCHAR(20)    NOT NULL,
    record_identifier   VARCHAR(200),
    field_name          VARCHAR(100),
    original_value      VARCHAR(500),
    corrected_value     VARCHAR(500),
    resolution_action   VARCHAR(50),
    details             VARCHAR(1000)
)
""").collect()

print("✓ DQ_LOG created")
print("  All transformation data quality issues will be logged here")

---

## Verification & Summary

Verify all infrastructure tables were created successfully.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Verification
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("VERIFICATION — Silver Infrastructure Tables")
print("=" * 70)

# List all tables in Silver schema
tables = session.sql(f"SHOW TABLES IN SCHEMA {SILVER_SCHEMA}").collect()

print(f"\nTables created in {SILVER_SCHEMA}:\n")
print(f"{'TABLE NAME':<40} {'ROW COUNT':>12}")
print("─" * 54)

infrastructure_tables = [
    'LKP_ORDER_STATUS',
    'LKP_PAYMENT_STATUS',
    'LKP_PAYMENT_METHOD',
    'LKP_LOYALTY_TIER',
    'LKP_WEIGHT_UNITS',
    'BRIDGE_CATEGORY',
    'BRIDGE_CUSTOMER_IDENTITY',
    'DQ_LOG'
]

for table_name in infrastructure_tables:
    try:
        count = session.table(f"{SILVER_SCHEMA}.{table_name}").count()
        status = "✓"
    except Exception as e:
        count = "ERROR"
        status = "✗"
    
    print(f"{status} {table_name:<38} {str(count):>12}")

print("─" * 54)
print("\n✓ Silver layer infrastructure setup complete!")
print("\nNext step: Run SILVER_INCREMENTAL_LOAD.ipynb to transform Bronze data")
print("=" * 70)